In [1]:
from datasets import Dataset, load_dataset
from transformers import PreTrainedTokenizerBase
from unsloth.chat_templates import standardize_sharegpt

from dev_util.weave_init import init_run
import training.train as tr


/home/ian-zhang/projects/tredoc/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
/home/ian-zhang/projects/tredoc/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ian-zhang/projects/tredoc/src/training/train.py:6: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
class ExampleDataLoader:
    def __init__(self):
        self.raw_data = load_dataset("mlabonne/FineTome-100k", split="train")

    def load_dataset_train(self):
        return Dataset.from_dict(self.raw_data[:90000])

    def load_dataset_validation(self):
        return Dataset.from_dict(self.raw_data[90000:])

    def format_dataset_training(self, dataset : Dataset, tokenizer : PreTrainedTokenizerBase):
        def formatting_prompts_func(examples):
            convos = examples["conversations"]
            texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
            return { "text" : texts, }

        dataset = standardize_sharegpt(dataset)
        dataset = dataset.map(formatting_prompts_func, batched=True)
        return dataset
    def format_dataset_inference(
        self, messages: list[tr.Conversation], tokenizer: PreTrainedTokenizerBase
    ):
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")
        return inputs
    def get_generation_sample(self):
        messages = [
            {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
        ]
        return messages


In [3]:
config = tr.get_config('example_config.yml')
run = init_run(config)
refit = tr.ModelFineTuner(config, ExampleDataLoader(), run)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ian-zhang/.netrc.
wandb: Currently logged in as: ianzhang (ianzhang-tredoc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Weave is installed but not imported. Add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


==((====))==  Unsloth 2026.3.5: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.552 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2379.59it/s]
Unsloth 2026.3.5 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.
Filter (num_proc=20): 100%|██████████| 90000/90000 [00:12<00:00, 6995.66 examples/s] 


Unsloth: Removed 91 out of 90000 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Filter (num_proc=20): 100%|██████████| 10000/10000 [00:03<00:00, 2945.09 examples/s]

Unsloth: Removed 3 out of 10000 samples from eval_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


In [4]:
refit.train()
refit.save()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 89,909 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
1,1.478068
2,0.745313
3,1.199118
4,1.172261
5,1.398121
6,1.078937
7,0.941773
8,0.435075
9,0.887751
10,1.157036


Unsloth: Will smartly offload gradients to save VRAM!


train/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/grad_norm,▅▄▄█▄▅▄▁▄▅▄▂▄▂▃▂▂▃▅▁▆▂▅▅█▆▂▇▁▂
train/learning_rate,▁▂▄▅▇██▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁
train/loss,▇▃▅▅▇▅▄▁▄▅▆▆▃▁▅▅▇▅▆▅▆▄▅▃█▇▄▄▄▅
total_flos,151638462424320.0
train/epoch,0.00133
train/global_step,30
train/grad_norm,0.44635
train/learning_rate,1e-05
train/loss,1.05554


wandb: Adding directory to artifact (/home/ian-zhang/projects/tredoc/final_model)... Done. 0.1s
Saving the dataset (1/1 shards): 100%|██████████| 90000/90000 [00:00<00:00, 148321.30 examples/s]
wandb: Adding directory to artifact (/home/ian-zhang/projects/tredoc/train_data)... Done. 1.0s
Saving the dataset (1/1 shards): 100%|██████████| 10000/10000 [00:00<00:00, 154466.99 examples/s]
wandb: Adding directory to artifact (/home/ian-zhang/projects/tredoc/eval_data)... Done. 0.1s
wandb: Adding directory to artifact (/home/ian-zhang/projects/tredoc/tokenizer)... Done. 0.0s
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/home/ian-zhang/projects/tredoc/.venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.mo

total_flos,151638462424320
train/epoch,0.00133
train/global_step,30
train/grad_norm,0.44635
train/learning_rate,1e-05
train/loss,1.05554
train_loss,1.09661
train_runtime,22.9611
train_samples_per_second,5.226
train_steps_per_second,1.307


In [6]:
text = refit.data_loader.format_dataset_inference(refit.data_loader.get_generation_sample(), refit.tokenizer)

In [9]:
from dataclasses import asdict

from unsloth import FastLanguageModel

FastLanguageModel.for_inference(refit.model)
refit.model.generate(**text, **asdict(refit.config.inference_config))

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RuntimeError: output with shape [1, 14, 1, 64] doesn't match the broadcast shape [1, 14, 53, 64]

In [13]:
model = FastLanguageModel.for_inference(refit.model)
result = model.generate(**text, use_cache=False, **asdict(refit.config.inference_config))

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
